In [1]:
from airflow.decorators import dag, task
from confluent_kafka import Producer
from faker import Faker
import logging
from datetime import datetime, timedelta
import boto3
from botocore.exceptions import ClientError
import json
import random  

fake = Faker()
logger = logging.getLogger(__name__)

def create_kafka_producer(config):
    """Create and return a Kafka producer."""
    return Producer(config)

def generate_log():
    """Generate synthetic log."""
    methods = ['GET', 'POST', 'PUT', 'DELETE']
    endpoints = ['/api/users', '/home', '/about', '/contact', '/services']
    statuses = [200, 301, 302, 400, 404, 500]
    
    user_agents = [
        'Mozilla/5.0 (iPhone; CPU iPhone OS 14_6 like Mac OS X)',
        'Mozilla/5.0 (X11; Linux x86_64)',
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
        'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    ]
    
    referrers = ['https://example.com', 'https://google.com', '-', 'https://bing.com', 'https://yahoo.com']
    
    ip = fake.ipv4()
    timestamp = datetime.now().strftime('%b %d %Y, %H:%M:%S')
    method = random.choice(methods)
    endpoint = random.choice(endpoints)
    status = random.choice(statuses)
    size = random.randint(1000, 15000)  
    referrer = random.choice(referrers)
    user_agent = random.choice(user_agents)
    
    log_entry = (
        f'{ip} - - [{timestamp}] "{method} {endpoint} HTTP/1.1" {status} {size} "{referrer}" "{user_agent}"'
    )
    return log_entry

def delivery_report(err, msg):
    """Callback for Kafka message delivery report."""
    if err is not None:
        logger.error(f'Message delivery failed: {err}')
    else:
        logger.info(f'Message delivered to {msg.topic()} [{msg.partition()}]')

def get_secret(secret_name, region_name='eu-north-1'):
    """Retrieve secrets from AWS Secrets Manager."""
    session = boto3.session.Session()
    client = session.client(service_name='secretsmanager', region_name=region_name)
    
    try:
        response = client.get_secret_value(SecretId=secret_name)
        return json.loads(response['SecretString'])
    except ClientError as e:
        logger.error(f"Secret retrieval failed. Error code: {e.response['Error']['Code']}, Message: {e.response['Error']['Message']}")
        raise

def produce_logs():
    """Produce log entries into Kafka."""
    secrets = get_secret('log_processing_key')
    kafka_config = {
        'bootstrap.servers': secrets['KAFKA_BOOTSTRAP_SERVER'],
        'security.protocol': 'SASL_SSL',
        'sasl.mechanisms': 'PLAIN',
        'sasl.username': secrets['KAFKA_SASL_USERNAME'],  
        'sasl.password': secrets['KAFKA_SASL_PASSWORD'],
        'session.timeout.ms': 50000
    }
    
    producer = create_kafka_producer(kafka_config)
    topic = 'website_logs'
    
    for _ in range(15000):
        log = generate_log()
        try:
            producer.produce(topic, log.encode('utf-8'), on_delivery=delivery_report)
        except Exception as e:
            logger.error(f'Error producing log: {e}')
            raise
    
    producer.flush()  # Flush after producing all messages
    logger.info(f'Produced 15,000 logs to topic {topic}')

default_args = {
    'owner': 'airflow',
    'depends_on_past': False,
    'email_on_failure': False,
    'retries': 1,
    'retry_delay': timedelta(seconds=5)
}

produce_logs()

[2025-02-10T10:47:15.091+0300] {credentials.py:1278} INFO - Found credentials in shared credentials file: ~/.aws/credentials


%4|1739173636.195|CONFWARN|rdkafka#producer-1| [thrd:app]: Configuration property session.timeout.ms is a consumer property and will be ignored by this producer instance
%6|1739173637.687|GETSUBSCRIPTIONS|rdkafka#producer-1| [thrd:main]: Telemetry client instance id changed from AAAAAAAAAAAAAAAAAAAAAA to d9xhkp6PT7mx+Bgxa6owZw


[2025-02-10T10:47:19.866+0300] {1744531638.py:52} INFO - Message delivered to website_logs [3]
[2025-02-10T10:47:19.867+0300] {1744531638.py:52} INFO - Message delivered to website_logs [3]
[2025-02-10T10:47:19.868+0300] {1744531638.py:52} INFO - Message delivered to website_logs [3]
[2025-02-10T10:47:19.869+0300] {1744531638.py:52} INFO - Message delivered to website_logs [3]
[2025-02-10T10:47:19.870+0300] {1744531638.py:52} INFO - Message delivered to website_logs [3]
[2025-02-10T10:47:19.870+0300] {1744531638.py:52} INFO - Message delivered to website_logs [3]
[2025-02-10T10:47:19.871+0300] {1744531638.py:52} INFO - Message delivered to website_logs [3]
[2025-02-10T10:47:19.872+0300] {1744531638.py:52} INFO - Message delivered to website_logs [3]
[2025-02-10T10:47:19.872+0300] {1744531638.py:52} INFO - Message delivered to website_logs [3]
[2025-02-10T10:47:19.873+0300] {1744531638.py:52} INFO - Message delivered to website_logs [3]
[2025-02-10T10:47:19.874+0300] {1744531638.py:52} 

In [11]:
from airflow.decorators import dag, task
from airflow.hooks.base import BaseHook
from confluent_kafka import Consumer, KafkaException
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk  
import logging
from datetime import datetime, timedelta
import boto3
from botocore.exceptions import ClientError
import json
import re  

logger = logging.getLogger(__name__)

def parse_log_entry(log_entry):
    """Parse a log entry using regex."""
    log_pattern = r'(?P<ip>[\d\.]+) - - \[(?P<timestamp>.*?)\] "(?P<method>\w+) (?P<endpoint>[\w/]+) (?P<protocol>[\w/\.]+)" (?P<status>\d+) (?P<size>\d+) "(?P<referrer>.*?)" "(?P<user_agent>.*?)"'
    match = re.match(log_pattern, log_entry)
    if not match:
        logger.warning(f"Invalid log format: {log_entry}")
        return None
    
    data = match.groupdict()
    
    try:
        parsed_timestamp = datetime.strptime(data['timestamp'], '%b %d %Y, %H:%M:%S')  
        data['@timestamp'] = parsed_timestamp.isoformat()
    except ValueError as e:
        logger.error(f"Timestamp parsing error: {data['timestamp']}. Error: {e}")
        return None
    
    return data

def create_kafka_consumer(config):
    """Create and return a Kafka consumer."""
    return Consumer(config)

def get_secret(secret_name, region_name='eu-north-1'):
    """Retrieve secrets from AWS Secrets Manager using Airflow connection."""
    aws_conn = BaseHook.get_connection("aws_id")  # Use the connection ID set in Airflow UI
    session = boto3.session.Session(
        aws_access_key_id=aws_conn.login,
        aws_secret_access_key=aws_conn.password,
        region_name=region_name,
    )
    client = session.client(service_name='secretsmanager', region_name=region_name)
    
    try:
        response = client.get_secret_value(SecretId=secret_name)
        return json.loads(response['SecretString'])
    except ClientError as e:
        logger.error(f"Secret retrieval failed. Error code: {e.response['Error']['Code']}, Message: {e.response['Error']['Message']}")
        raise
    
def consume_and_index_logs():
    """Consume log entries from Kafka and index them into Elasticsearch."""
    secrets = get_secret('log_processing_key')
    
    consumer_config = {
        'bootstrap.servers': secrets['KAFKA_BOOTSTRAP_SERVER'],
        'security.protocol': 'SASL_SSL',
        'sasl.mechanisms': 'PLAIN',
        'sasl.username': secrets['KAFKA_SASL_USERNAME'],  
        'sasl.password': secrets['KAFKA_SASL_PASSWORD'],
        'group.id': 'web_log_processing',
        'auto.offset.reset': 'latest'  
    }
    
    es_config = {
        'hosts': [secrets['ELASTICSEARCH_URL']],
        'http_auth': ('api_key', secrets['ELASTICSEARCH_API_KEY'])  
    }
    
    consumer = create_kafka_consumer(consumer_config)
    es = Elasticsearch(**es_config)
    topic = 'website_logs'
    consumer.subscribe([topic])
    
    try:
        logs_index = 'log_processing'
        if not es.indices.exists(index=logs_index):
            es.indices.create(index=logs_index, body={})  
            logger.info(f'Created index: {logs_index}')
    except Exception as e:
        logger.error(f'Failed to create index: {logs_index}. Error: {e}')
        
    try:
        logs = []
        while True:
            msg = consumer.poll(timeout=1.0)  
            if msg is None:
                continue
            
            if msg.error():
                if msg.error().code() == KafkaException._PARTITION_EOF:
                    break
                raise KafkaException(msg.error())
            
            log_entry = msg.value().decode('utf-8')
            parsed_log = parse_log_entry(log_entry)
            
            if parsed_log:
                logs.append(parsed_log)
                
            # Index when 15000 logs are collected
            if len(logs) >= 15000:
                actions = [
                    {
                        '_op_type': 'index',
                        '_index': logs_index,
                        '_source': log
                    }
                    for log in logs
                ]
                success, failed = bulk(es, actions, refresh=True)
                logger.info(f'Indexed {success} logs, {len(failed)} failed')
                logs = []       
    except Exception as e:
        logger.error(f'Failed to index logs: {e}')
        
    # Index any remaining logs
    try:
        if logs:
            actions = [
                {
                    '_op_type': 'index',
                    '_index': logs_index,
                    '_source': log
                }
                for log in logs
            ]
            success, failed = bulk(es, actions, refresh=True)
            logger.info(f'Indexed remaining {success} logs, {len(failed)} failed')
    except Exception as e:
        logger.error(f'Log processing error: {e}')
    finally:
        consumer.close()
        es.close()
           
default_args = {
    'owner': 'airflow',
    'depends_on_past': False,
    'email_on_failure': False,
    'retries': 1,
    'retry_delay': timedelta(seconds=5)
}

@dag(
    'logs_consuming_pipeline',
    description='Consume synthetic logs',
    start_date=datetime(2025, 1, 1),
    schedule="@daily",
    catchup=False,
    default_args=default_args,
    tags=["log_consuming"],
)
def logs_consuming_pipeline():
    @task
    def consume_and_index_logs_task():
        """Task to consume logs from Kafka and index them into Elasticsearch."""
        consume_and_index_logs()

    consume_and_index_logs_task()

# Instantiate the DAG
dag_instance = logs_consuming_pipeline()


In [2]:
from elasticsearch import Elasticsearch

es = Elasticsearch("https://948f0aa0871f4f30ac4126713fe4b67d.eu-north-1.aws.elastic-cloud.com:9243")

query = {
    "query": {
        "match_all": {}
    }
}

response = es.search(index="web_logs", body=query)
print(response)


AuthenticationException: AuthenticationException(401, 'security_exception', 'missing authentication credentials for REST request [/web_logs/_search]')